# Example Usage of SpliceSeqExtractor Class

This notebook is intended to showcase the `SpliceSeqExtractor` class and how it might be beneficial for motif discovery! Note that `logomaker` is not a required dependency of this repo, but you can install it using `pip install logomaker`. This package is solely for visualizing the motifs that we find from our sequences. Additionally `itertools` is a built-in python package that contains some useful functions -- in this case, I'm just using `islice` to easily grab a couple of values from a dictionary.

In [ ]:
from itertools import islice
from Bio.Seq import Seq
import logomaker
from protisplice import SpliceSeqExtractor, ExtractionParams, JunctionType
from protisplice import generate_pfm, generate_ppm, generate_pwm
from protisplice import write_sequences

## About the Dataset

For the data, I used human chromosome 1 (GENCODE FASTA + annotations). This class should work with any GFF3 and FASTA from any species, provided that you're using the same version numbers for both files (else the annotations won't match the proper coordinates). Of course, annotation quality is going to vary between species, especially for those that aren't very well-studied. 

GENCODE doesn't provide per-chromosome annotations as separate files, so I will include some simple bash scripts that I used to split out the whole genome by chromosome, located in `protisplice/bash_utils`.

## Initialization and Attributes of SpliceSeqExtractor

First, we can instantiate the class. Note that the GFF3 path and FASTA paths are required arguments, and the transcript filter is optional. By default, the transcript filter is `"all"`, but we can filter by transcripts that are considered protein coding by the annotation file with `"protein_coding"`. 

In [ ]:
params = ExtractionParams(n_exon=10, n_intron=10)
extractor = SpliceSeqExtractor(gff_path="data/input/chr1.gff3", fasta_path="data/input/chr1.fa", transcript_filter="protein_coding", extraction_params=params)


The extractor has two properties - `junctions` and `transcripts`. These are both lazy-loaded, so they will be computed upon any method that requires these properties. First, we can take a look at the identified junctions:

In [ ]:
junctions = extractor.junctions

The junctions are generated using a `SpliceJunction` object, which essentially is just a dictionary containing the transcript ID, sequence ID (usually going to be the chromosome # for most purposes), the coordinate of the junction, strand, and the type of junction - either donor or acceptor.

Note that the junction coordinates are 1-based and represent the first exon base along the junction (i.e. coord - 1 is the end of the intron). 

Lastly, the transcript ID is suffixed with ID_junctype_num, where junctype is either donor/acceptor, and num is the sequential order in which each splice junction is found, starting at 0. 

In [ ]:
junctions[:5]

### Quick Note about Alternative Splicing

Currently, I have not implemented functionality to identify certain transcripts that are alternatively spliced, so it is possible some donor or acceptor junctions are actually unused. I may add RNA-seq validation in later, but this would be a pretty big undertaking!

We can also take a look at all of the transcripts found using `extractor.transcripts`, which will return a `Transcript` object, a dictionary of dictionaries. `Transcript.info` contains the sequence ID from the associated FASTA and the strand type. 

`Transcript.exons` contains the 1-based start and end coordinates for every exon in the transcript. `Transcript.introns` is initiated to `None` but can be determined using `Transcript.get_intron_coords`. 

Reason for this being that the introns can already be inferred from the exon coordinates, and it would take up more memory to explicitly hold the intron coordinates as well, but the method exists if you need it.

In [ ]:
transcripts = extractor.transcripts
dict(islice(transcripts.items(), 5))

Lastly, we will get to the bigger workflow functions from this class. `extract_splice_sites` will return a JunctionData object containing the splice junction, window start, window end, and the sequence.

In [ ]:
true_ss = extractor.sequence_extractor.extract_splice_sites(junctions)
true_ss[:5]

In [ ]:
true_ss[1].sequence

## Motif Discovery and Visualization for Splice Sites

To visualize the different motifs between donors and acceptors, we'll split the sequences based on their junction type into two lists using a simple list comprehension. 

In [ ]:
acceptors = [true_ss[x].sequence for x in range(len(true_ss)) if true_ss[x].junction.junction_type == JunctionType.ACCEPTOR]
# acceptors[:5]

donors = [true_ss[x].sequence for x in range(len(true_ss)) if true_ss[x].junction.junction_type == JunctionType.DONOR]
donors[:5]

Now we can see a cool use case for the SpliceSeqExtractor class! In addition to the sequence extraction methods, I have also provided methods to generate position frequency, weight, and probability matrices from a list of sequences. All of these methods will return a pandas DataFrame, where the columns are the nucleotides A, T, G, C, and the values represent either the count, log-odds ratio, or the probability of each nucleotide at each position.  

In this case, I've used the position probability matrices to visualize the acceptor and donor motifs for human chromosome 1.

In [ ]:
donors_seqs = [Seq(s) for s in donors]
acceptors_seqs = [Seq(s) for s in acceptors]

ppm_donors = generate_ppm(donors_seqs)
ppm_acceptors = generate_ppm(acceptors_seqs)

Now, we can pass in our dataframes into `logomaker` to visualize our motifs!

In [ ]:
logomaker.Logo(ppm_donors, color_scheme='classic')
logomaker.Logo(ppm_acceptors, color_scheme='classic')

## Dataset Generation for Predicting Splice Sites

In addition to just looking at our motifs, we can also use the extractor's random sampling methods to generate higher-quality "decoy" sequences. For model training, we want to maximize the variety of the types of sequences we train on. Using solely intergenic sequences might lack certain structural features of introns/exons, and essentially make it too easy to discern splice sites versus non-splice sites. 

For example, if we were classifying images of cats, we wouldn't want to train the model with pictures of grass for our non-cat pictures.

To create more generalizable models, the extractor has a couple of methods for sampling regions close to splice sites (in introns and exons), but exclude the splice site itself. Users can pass in a buffer size to start sampling within a certain region away from the junction. Users are also able to sample random intergenic regions as well, if desired.

In [ ]:
intergenic_seqs = extractor.extract_intergenic_sequences(target_count=5)
exonic_seqs = extractor.extract_exonic_sequences(target_count=5)
intronic_seqs = extractor.extract_intronic_regions(target_count=5)

### Writing Sequences to FASTA

Lastly, you can write out the extracted sequences to FASTA simply using the `write_sequences()` function, which takes in a list of `JunctionData` objects. These objects are the output of all of the extraction methods. All of the important metadata about the `JunctionData` objects are retained in the FASTA header in the format `>seqid_seqtype_strand_winstart_winend`:

In [ ]:
count = write_sequences(exonic_seqs, "exonic_decoy_seqs.fasta")

exonic_seqs[0].to_fasta_header()

## Ending Notes and Other Use Cases

With the sequences written out into separate files, users are able to create their own ML pipelines for splice site classification! Hopefully anyone interested in training ML models finds this repo useful. If you have any questions or issues, please open an issue and I'll get back to you ASAP. 

I plan to make this package more performant in the future by adding multithreading support for certain operations. However, I was able to extract all of the splice sites from chromosome 1 on my MacBook Pro, M4 Pro, 24GB RAM in ~7 seconds with the current implementation. 